# Finite-strain topology optimization

This notebook first reproduces the C-shape fictitious-domain test in Figs. 1–3
of Wang et al. (2014), then reproduces the 144 kN, 240 kN, and 300 kN
cantilever examples in Fig. 5. Low-density elements use the proposed energy
interpolation between small- and finite-deformation elasticity.

> Wang, F., Lazarov, B. S., Sigmund, O., and Jensen, J. S. “Interpolation
> scheme for fictitious domain techniques and topology optimization of finite
> strain elastic problems.” *Computer Methods in Applied Mechanics and
> Engineering* 276 (2014): 453–472.

In [ ]:
import jax
import jax.numpy as np

jax.config.update("jax_enable_x64", True)

import io
from contextlib import redirect_stdout
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as onp
from IPython.display import Image as DisplayImage
from IPython.display import display
from jax_fem import logger
from PIL import Image as PILImage

logger.setLevel("WARNING")

from jax_fem.generate_mesh import Mesh, get_meshio_cell_type, rectangle_mesh
from jax_fem.solver import ad_wrapper, solver

from topax.filter import build_conv_filter
from topax.optimizer import MMA
from topax.problem import TopOptProblem
from topax.projection import proj_fn_wang

## 1. C-shape fictitious-domain validation

The C-shaped solid is embedded in a `10 × 10` fixed mesh. The direct model
uses finite-deformation elements in both solid and void regions; the
interpolated model uses nearly linear kinematics in the low-density region.

In [ ]:
class CShape(TopOptProblem):
    """St. Venant-Kirchhoff C-shape with optional energy interpolation."""

    def custom_init(self, use_interp):
        self.fe = self.fes[0]
        self.use_interp = use_interp
        self.t_top = 0.0
        self.t_bot = 0.0

        Lx = np.max(self.fe.points[:, 0])
        Ly = np.max(self.fe.points[:, 1])

        def top_right(point):
            return np.logical_and(
                np.isclose(point[0], Lx, atol=1e-5),
                np.isclose(point[1], Ly, atol=1e-5),
            )

        def bottom_right(point):
            return np.logical_and(
                np.isclose(point[0], Lx, atol=1e-5),
                np.isclose(point[1], 0., atol=1e-5),
            )

        self.add_point_load(
            top_right, lambda point: np.array([0., -self.t_top])
        )
        self.add_point_load(
            bottom_right, lambda point: np.array([self.t_bot, 0.])
        )

    def set_params(self, params):
        rho, self.t_top, self.t_bot = params
        self.internal_vars = [
            np.repeat(rho[None], self.fe.num_quads, axis=0).transpose(1, 0, 2)
        ]

    def get_tensor_map(self):
        def stress(u_grad, theta):
            rho = theta[0]
            penal = 3.0
            E = 1e-9 + rho**penal * (1.0 - 1e-9)
            nu = 0.3

            mu = E / (2. * (1. + nu))
            lam = E * nu / ((1. + nu) * (1. - 2. * nu))
            lam = 2. * mu * lam / (lam + 2. * mu)

            def PK1_stress(F):
                E_gl = 0.5 * (F.T @ F - np.eye(F.shape[-1]))
                S = lam * np.trace(E_gl) * np.eye(F.shape[-1]) + 2. * mu * E_gl
                return F @ S

            if self.use_interp:
                gamma = proj_fn_wang(rho**penal, beta=500.0, eta=0.01)
                I = np.eye(u_grad.shape[-1])
                eps = 0.5 * (u_grad + u_grad.T)
                sigma = lam * np.trace(eps) * I + 2.0 * mu * eps
                u_grad_gamma = gamma * u_grad
                eps_gamma = 0.5 * (u_grad_gamma + u_grad_gamma.T)
                sigma_gamma = (
                    lam * np.trace(eps_gamma) * I + 2.0 * mu * eps_gamma
                )
                return PK1_stress(I + u_grad_gamma) - sigma_gamma + sigma
            return PK1_stress(np.eye(u_grad.shape[-1]) + u_grad)

        return stress


def prep_cshape(Nx, Ny, Lx, Ly, interp=True):
    ele_type = 'QUAD4'
    cell_type = get_meshio_cell_type(ele_type)
    meshio_mesh = rectangle_mesh(Nx=Nx, Ny=Ny, domain_x=Lx, domain_y=Ly)
    mesh = Mesh(meshio_mesh.points, meshio_mesh.cells_dict[cell_type])

    def fixed_location(point):
        return np.isclose(point[0], 0., atol=1e-5)

    problem = CShape(
        mesh,
        vec=2,
        dim=2,
        ele_type=ele_type,
        dirichlet_bc_info=[
            [fixed_location] * 2,
            [0, 1],
            [lambda point: 0.] * 2,
        ],
        additional_info=(interp,),
    )

    fwd_pred = ad_wrapper(
        problem,
        solver_options={'newton': {'linear': {'spsolve_solver': {}}}},
        adjoint_solver_options={'spsolve_solver': {}},
    )

    return fwd_pred, meshio_mesh.points, problem

In [ ]:
# C-SHAPE FORWARD TEST
Nx_c, Ny_c = 10, 10
Lx_c, Ly_c = 10.0, 10.0

rho_c = onp.ones((Nx_c, Ny_c))
rho_c[1:, 1:-1] = 1e-9
rho_c = onp.ascontiguousarray(rho_c.flatten()[:, None])

cshape_cases = [
    (r'f_1=0.002,\ f_2=0.003', 0.002, 0.003),
    (r'f_1=0.018,\ f_2=0.027\ (9\times)', 0.018, 0.027),
]

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
plot_options = {
    'cmap': 'gray_r',
    'vmin': 0,
    'vmax': 1,
    'edgecolors': 'k',
    'linewidth': 0.3,
}

for column, interp in enumerate([False, True]):
    name = r'\mathrm{Direct}' if not interp else r'\mathrm{Interp}'
    fwd_pred, points, _ = prep_cshape(
        Nx_c, Ny_c, Lx_c, Ly_c, interp=interp
    )
    for row, (label, t_top, t_bot) in enumerate(cshape_cases):
        solution = fwd_pred((rho_c, t_top, t_bot))[0]
        ax = axes[row, column]
        X = points[:, 0].reshape(Nx_c + 1, Ny_c + 1).T
        Y = points[:, 1].reshape(Nx_c + 1, Ny_c + 1).T
        C = rho_c[:, 0].reshape(Nx_c, Ny_c).T
        X += solution[:, 0].reshape(Nx_c + 1, Ny_c + 1).T
        Y += solution[:, 1].reshape(Nx_c + 1, Ny_c + 1).T
        ax.pcolormesh(X, Y, C, **plot_options)
        ax.axis('equal')
        ax.set_axis_off()
        ax.set_title(rf'${name}\;({label})$', fontsize=22, pad=12)

plt.tight_layout(h_pad=1.8)
output_path = Path('docs/imgs/example_topopt_nlgeo_cshape.png')
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()

## 2. Cantilever topology optimization

The cantilever cases use the same energy interpolation, density filtering,
projection, incremental loading, and continuation strategy described in the
paper.

### 2.1 Finite-element model

In [ ]:
class Hyperelastic(TopOptProblem):
    """St. Venant-Kirchhoff solid with Wang et al. energy interpolation."""

    def custom_init(self):
        self.fe = self.fes[0]
        self.fe.flex_inds = np.arange(len(self.fe.cells))
        self.load = 0.0

        Lx = np.max(self.fe.points[:, 0])
        Ly = np.max(self.fe.points[:, 1])

        def load_location(point):
            return np.logical_and(
                np.isclose(point[0], Lx, atol=1e-5),
                np.isclose(point[1], Ly / 2., atol=1e-5),
            )

        self.load_node = self.add_point_load(
            load_location,
            lambda point: np.array([0., -self.load]),
        )

    def get_tensor_map(self):
        def stress(u_grad, theta, penal):
            rho = theta[0]
            stiffness = 1e-9 + rho**penal * (1.0 - 1e-9)

            nu = 0.4
            mu = 1.0 / (2.0 * (1.0 + nu))
            lam = nu / ((1.0 + nu) * (1.0 - 2.0 * nu))

            def linear_stress(grad):
                strain = 0.5 * (grad + grad.T)
                return lam * np.trace(strain) * np.eye(self.dim) + 2.0 * mu * strain

            gamma = proj_fn_wang(rho**penal, beta=500.0, eta=0.01)
            scaled_grad = gamma * u_grad
            deformation_gradient = np.eye(self.dim) + scaled_grad
            green_strain = 0.5 * (
                deformation_gradient.T @ deformation_gradient - np.eye(self.dim)
            )
            second_pk = (
                lam * np.trace(green_strain) * np.eye(self.dim)
                + 2.0 * mu * green_strain
            )
            first_pk = deformation_gradient @ second_pk

            return stiffness * (
                linear_stress(u_grad)
                + gamma * (first_pk - linear_stress(scaled_grad))
            )

        return stress

    def set_params(self, params):
        density, penal, load = params
        full_density = np.ones((self.fe.num_cells, density.shape[1]))
        full_density = full_density.at[self.fe.flex_inds].set(density)
        density_quads = np.repeat(
            full_density[:, None, :], self.fe.num_quads, axis=1
        )
        penal_quads = penal * np.ones((self.fe.num_cells, self.fe.num_quads))

        self.full_params = full_density
        self.internal_vars = [density_quads, penal_quads]
        self.load = load


def prep_fem(Nx, Ny, Lx, Ly):
    ele_type = 'QUAD4'
    cell_type = get_meshio_cell_type(ele_type)
    meshio_mesh = rectangle_mesh(Nx=Nx, Ny=Ny, domain_x=Lx, domain_y=Ly)
    mesh = Mesh(meshio_mesh.points, meshio_mesh.cells_dict[cell_type])

    def fixed_location(point):
        return np.isclose(point[0], 0., atol=1e-5)

    zero = lambda point: 0.
    problem = Hyperelastic(
        mesh,
        vec=2,
        dim=2,
        ele_type=ele_type,
        dirichlet_bc_info=[
            [fixed_location, fixed_location],
            [0, 1],
            [zero, zero],
        ],
    )
    return problem

### 2.2 Numerical settings and incremental loading

In [ ]:
# COMMON SETTINGS
vf = 0.5
Nx, Ny = 120, 30
Lx, Ly = 1.0, 0.25
rmin = 3.75

E1 = 3.0e9
thickness = 0.10

linear_options = {'spsolve_solver': {}}
newton_options = {
    'newton': {
        'tol': 1e-6,
        'rel_tol': 1e-8,
        'line_search_flag': True,
        'linear': linear_options,
    }
}


def physical_density(x, H, Hs, beta):
    x_col = x.reshape(-1, 1)
    x_tilde = ((H @ x_col) / Hs).reshape(x.shape)
    return proj_fn_wang(x_tilde, beta=beta, eta=0.5)


def volume_constraint(x, H, Hs, beta):
    return np.mean(physical_density(x, H, Hs, beta)) / vf - 1.0


def normalized_load(load_kn):
    return load_kn * 1e3 / (E1 * thickness)


def incremental_solve(
    problem,
    density,
    penal,
    final_load,
    num_steps,
    previous_history=None,
):
    zero_solution = [
        np.zeros((problem.fe.num_total_nodes, problem.fe.vec))
    ]
    solution = zero_solution
    history = []

    for step, load in enumerate(np.linspace(final_load / num_steps, final_load, num_steps)):
        if previous_history is None:
            initial_guess = solution
        else:
            initial_guess = previous_history[step]

        problem.set_params((density, penal, load))
        options = {
            'newton': {
                **newton_options['newton'],
                'initial_guess': initial_guess,
            }
        }
        solution = solver(problem, options)
        history.append(solution)

    return solution, history


def evaluate_design(
    problem,
    x,
    penal,
    beta,
    load_kn,
    H,
    Hs,
    previous_history,
):
    load = normalized_load(load_kn)
    num_steps = int(onp.ceil(load_kn / 10.0))
    density = physical_density(x, H, Hs, beta)

    equilibrium, history = incremental_solve(
        problem,
        density,
        penal,
        load,
        num_steps,
        previous_history,
    )

    fwd_pred = ad_wrapper(
        problem,
        solver_options={
            'newton': {
                **newton_options['newton'],
                'initial_guess': equilibrium,
            }
        },
        adjoint_solver_options=linear_options,
    )

    def objective(design):
        density = physical_density(design, H, Hs, beta)
        displacement = fwd_pred((density, penal, load))[0]
        compliance_kj = -load_kn * displacement[problem.load_node, 1]
        return compliance_kj, (density, displacement)

    (objective_value, (density, displacement)), objective_grad = (
        jax.value_and_grad(objective, has_aux=True)(x)
    )
    constraint_value, constraint_grad = jax.value_and_grad(
        lambda design: volume_constraint(design, H, Hs, beta)
    )(x)

    return (
        objective_value,
        objective_grad,
        constraint_value,
        constraint_grad,
        density,
        displacement,
        history,
    )

### 2.3 MMA optimization and continuation

In [ ]:
def run_optimization(load_kn, max_iterations=260, frame_stride=2, move=0.1):
    problem = prep_fem(Nx, Ny, Lx, Ly)
    H, Hs = build_conv_filter(problem, rmin=rmin)
    optimizer = MMA(move=move)

    x = vf * np.ones((Nx * Ny, 1))
    penal = 1.0
    beta = 4.0
    penal_counter = 0
    beta_counter = 0
    previous_history = None
    frames = []
    records = []

    for iteration in range(1, max_iterations + 1):
        (
            objective,
            objective_grad,
            constraint,
            constraint_grad,
            density,
            displacement,
            previous_history,
        ) = (None,) * 7
        with redirect_stdout(io.StringIO()):
            (
                objective,
                objective_grad,
                constraint,
                constraint_grad,
                density,
                displacement,
                previous_history,
            ) = evaluate_design(
                problem,
                x,
                penal,
                beta,
                load_kn,
                H,
                Hs,
                previous_history,
            )

        penal_used = penal
        beta_used = beta
        xold = x.copy()
        x = np.asarray(
            optimizer.update(
                xold,
                objective,
                objective_grad,
                constraint,
                constraint_grad,
            )
        )
        change = float(np.max(np.abs(x - xold)))
        volume = float(np.mean(density))

        converged = (
            penal_used >= 3.0
            and beta_used >= 64.0
            and change <= 0.01
        )

        continuation_changed = False
        if not converged:
            if penal < 3.0:
                penal_counter += 1
                interval = 2 if penal < 2.0 else 5
                if penal_counter >= interval:
                    penal = min(penal + 0.05, 3.0)
                    penal_counter = 0
                    continuation_changed = True
                    if penal >= 3.0:
                        beta_counter = 0
            elif beta < 64.0:
                beta_counter += 1
                if beta_counter >= 10:
                    beta = min(2.0 * beta, 64.0)
                    beta_counter = 0
                    continuation_changed = True

        record = {
            'iteration': iteration,
            'objective': float(objective),
            'volume': volume,
            'change': change,
            'penal': float(penal_used),
            'beta': float(beta_used),
        }
        records.append(record)
        print(
            f"load={load_kn:3.0f} kN, It.:{iteration:4d}, "
            f"Obj.:{float(objective):10.4f} kJ, Vol.:{volume:7.3f}, "
            f"ch.:{change:7.3f}, p={penal_used:4.2f}, beta={beta_used:4.0f}"
        )

        if (
            iteration == 1
            or iteration % frame_stride == 0
            or continuation_changed
        ):
            frames.append(
                {
                    **record,
                    'density': onp.asarray(density),
                    'displacement': onp.asarray(displacement),
                }
            )

        if converged:
            break

    # Re-evaluate the accepted final design for the final image and objective.
    (
        objective,
        _,
        constraint,
        _,
        density,
        displacement,
        previous_history,
    ) = (None,) * 7
    with redirect_stdout(io.StringIO()):
        (
            objective,
            _,
            constraint,
            _,
            density,
            displacement,
            previous_history,
        ) = evaluate_design(
            problem,
            x,
            penal,
            beta,
            load_kn,
            H,
            Hs,
            previous_history,
        )
    final_record = {
        'iteration': len(records),
        'objective': float(objective),
        'volume': float(np.mean(density)),
        'change': records[-1]['change'],
        'penal': float(penal),
        'beta': float(beta),
        'density': onp.asarray(density),
        'displacement': onp.asarray(displacement),
    }
    frames.append(final_record)

    return {
        'load_kn': load_kn,
        'problem': problem,
        'design': x,
        'frames': frames,
        'records': records,
        'final': final_record,
    }

### 2.4 Run the load cases

In [ ]:
# RUN THE THREE REPRODUCED FIG. 5 LOAD CASES
case_144 = run_optimization(144.0)
case_240 = run_optimization(240.0)
case_300 = run_optimization(300.0)

### 2.5 Save results and render deformed-topology animations

In [ ]:
def save_case_data(case, filename):
    output_path = Path('docs/data') / filename
    output_path.parent.mkdir(parents=True, exist_ok=True)

    frames = case['frames']
    records = case['records']
    onp.savez_compressed(
        output_path,
        load_kn=case['load_kn'],
        points=onp.asarray(case['problem'].fe.points),
        cells=onp.asarray(case['problem'].fe.cells),
        design=onp.asarray(case['design']),
        frame_density=onp.stack([frame['density'] for frame in frames]),
        frame_displacement=onp.stack(
            [frame['displacement'] for frame in frames]
        ),
        frame_iteration=onp.array([frame['iteration'] for frame in frames]),
        frame_objective=onp.array([frame['objective'] for frame in frames]),
        frame_volume=onp.array([frame['volume'] for frame in frames]),
        frame_change=onp.array([frame['change'] for frame in frames]),
        frame_penal=onp.array([frame['penal'] for frame in frames]),
        frame_beta=onp.array([frame['beta'] for frame in frames]),
        record_iteration=onp.array([record['iteration'] for record in records]),
        record_objective=onp.array([record['objective'] for record in records]),
        record_volume=onp.array([record['volume'] for record in records]),
        record_change=onp.array([record['change'] for record in records]),
        record_penal=onp.array([record['penal'] for record in records]),
        record_beta=onp.array([record['beta'] for record in records]),
    )
    return output_path


def case_points(case):
    if 'points' in case:
        return case['points']
    return onp.asarray(case['problem'].fe.points)


def global_deformation_bounds(cases):
    all_x = []
    all_y = []
    for case in cases:
        points = case_points(case)
        for frame in case['frames']:
            deformed = points + frame['displacement']
            all_x.append(deformed[:, 0])
            all_y.append(deformed[:, 1])

    xmin = min(onp.min(values) for values in all_x)
    xmax = max(onp.max(values) for values in all_x)
    ymin = min(onp.min(values) for values in all_y)
    ymax = max(onp.max(values) for values in all_y)
    margin = 0.01 * max(xmax - xmin, ymax - ymin)
    return xmin - margin, xmax + margin, ymin - margin, ymax + margin


def render_deformed_frame(case, frame, bounds):
    points = case_points(case)
    displacement = frame['displacement']
    density = frame['density'].reshape(Ny, Nx, order='F')

    x_deformed = (
        points[:, 0] + displacement[:, 0]
    ).reshape(Nx + 1, Ny + 1).T
    y_deformed = (
        points[:, 1] + displacement[:, 1]
    ).reshape(Nx + 1, Ny + 1).T

    with plt.rc_context({
        'text.usetex': True,
        'font.family': 'serif',
        'font.size': 16,
    }):
        fig = plt.figure(figsize=(10, 5.6), dpi=150)
        ax = fig.add_axes([0.025, 0.035, 0.95, 0.84])
        ax.pcolormesh(
            x_deformed,
            y_deformed,
            density,
            cmap='gray_r',
            vmin=0,
            vmax=1,
            shading='flat',
            edgecolors='0.90',
            linewidth=0.02,
        )
        ax.set_xlim(bounds[0], bounds[1])
        ax.set_ylim(bounds[2], bounds[3])
        ax.set_aspect('equal', adjustable='box')
        ax.axis('off')
        fig.suptitle(
            rf"$f={case['load_kn']:.0f}\,\mathrm{{kN}}\quad "
            rf"i={frame['iteration']}\quad "
            rf"c={frame['objective']:.3f}\,\mathrm{{kJ}}\quad "
            rf"V={frame['volume']:.3f}\quad "
            rf"p={frame['penal']:.2f}\quad "
            rf"\beta={frame['beta']:.0f}$",
            y=0.955,
        )
        fig.canvas.draw()
        rgba = onp.asarray(fig.canvas.buffer_rgba()).copy()
        plt.close(fig)
    return PILImage.fromarray(rgba)


def save_deformed_gif(case, filename, bounds):
    images = [
        render_deformed_frame(case, frame, bounds)
        for frame in case['frames']
    ]
    output_path = Path('docs/imgs') / filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    images[0].save(
        output_path,
        save_all=True,
        append_images=images[1:],
        duration=120,
        loop=0,
        optimize=False,
    )
    display(DisplayImage(filename=str(output_path)))


cases = [case_144, case_240, case_300]
shared_bounds = global_deformation_bounds(cases)
for case in cases:
    load = int(case['load_kn'])
    save_case_data(case, f'example_topopt_nlgeo_{load}kn.npz')
    save_deformed_gif(
        case,
        f'example_topopt_nlgeo_{load}kn.gif',
        shared_bounds,
    )


### 2.6 Comparison with Fig. 5

In [ ]:
reference_objectives = {
    144: 21.2747,
    240: 56.9401,
    300: 84.9415,
}

for case in [case_144, case_240, case_300]:
    load = int(case['load_kn'])
    result = case['final']['objective']
    reference = reference_objectives[load]
    error = 100.0 * (result - reference) / reference
    print(
        f"{load} kN: objective={result:.4f} kJ, "
        f"reference={reference:.4f} kJ, error={error:+.2f}%, "
        f"volume={case['final']['volume']:.4f}, "
        f"iterations={len(case['records'])}"
    )